# Ejecutar el proyecto en Kaggle

Este notebook se copia a Kaggle y ejecuta los notebooks del repositorio en orden. Cada notebook queda guardado con sus resultados visibles y todo se sube al repositorio al final.

**Antes de empezar**
1. Crear el notebook desde la pestaña **Code** de la competencia *Understanding Clouds from Satellite Images*, para que traiga los datos.
2. En **Session options** activar **Internet** y elegir **GPU T4 x2**.
3. En **Add-ons → Secrets** crear `GITHUB_TOKEN` con un token clásico de GitHub con permiso `repo`.
4. Cambiar `REPO` en la primera celda por el repositorio propio.

Los notebooks de modelos se ejecutan de a dos, uno en cada GPU. En la corrida anterior, cada modelo tardó alrededor de una hora.

**Para dejarlo corriendo sin el navegador abierto**, usar **Save Version → Save & Run All (Commit)**. Kaggle ejecuta todo en segundo plano, hasta 12 horas, y la última celda sube los resultados al repositorio.

## 1. Clonar el repositorio

In [ ]:
from kaggle_secrets import UserSecretsClient
token = UserSecretsClient().get_secret("GITHUB_TOKEN")
REPO = "Nepthunders/NOMBRE_DEL_REPO"
!git clone -q https://{token}@github.com/{REPO}.git /kaggle/working/repo
!git -C /kaggle/working/repo remote set-url origin https://github.com/{REPO}.git
!ls /kaggle/working/repo

## 2. Entorno y preparación de datos (unos 5 minutos)

In [ ]:
%cd /kaggle/working/repo
!git config user.name "Nepthunders"
!git config user.email "SU_CORREO_DE_GITHUB"
!pip install -q segmentation-models-pytorch
!python -c "import sys; sys.path.insert(0, 'src'); from datos import preparar_datos; preparar_datos()"
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 3. Funciones para ejecutar notebooks
`ejecutar()` corre un notebook completo con `nbconvert` y lo guarda con sus resultados, en la GPU indicada.

In [ ]:
import os
import subprocess
import time
from pathlib import Path

import pandas as pd

os.chdir("/kaggle/working/repo")


def ejecutar(notebook, gpu=None):
    env = dict(os.environ)
    if gpu is not None:
        env["CUDA_VISIBLE_DEVICES"] = str(gpu)
    log = open(f"/kaggle/working/log_{notebook}.txt", "w")
    cmd = ["jupyter", "nbconvert", "--to", "notebook", "--execute", "--inplace",
           "--ExecutePreprocessor.timeout=-1", "--ExecutePreprocessor.kernel_name=python3", f"notebooks/{notebook}"]
    return subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, env=env)


def esperar(proc, notebook):
    codigo = proc.wait()
    print(f"{notebook} terminó con código {codigo}")
    if codigo != 0:
        print(open(f"/kaggle/working/log_{notebook}.txt").read()[-3000:])

## 4. EDA
Corre en CPU. Tarda unos minutos porque recorre las 5546 imágenes.

In [ ]:
p = ejecutar("00_eda.ipynb")
esperar(p, "00_eda.ipynb")

## 5. Modelos
Se ejecutan de a dos, uno por GPU. Cuando uno termina, entra el siguiente en la GPU que quedó libre. Cada 5 minutos se imprime la última época de cada modelo. Para correr solo algunos, deja en `NOTEBOOKS` únicamente esos.

In [ ]:
NOTEBOOKS = {
    "03_unet_r34.ipynb": "unet_r34",
    "04_deeplab_r34.ipynb": "deeplab_r34",
    "02_att_resunet.ipynb": "att_resunet",
    "01_cnn.ipynb": "cnn",
}
cola = list(NOTEBOOKS)
activos = {}
for gpu in (0, 1):
    if cola:
        nb = cola.pop(0)
        activos[gpu] = (nb, ejecutar(nb, gpu))

while activos:
    time.sleep(300)
    for gpu, (nb, proc) in list(activos.items()):
        hist = Path(f"artifacts/{NOTEBOOKS[nb]}/historial.csv")
        if hist.exists() and len(pd.read_csv(hist)):
            u = pd.read_csv(hist).iloc[-1]
            estado = f"época {int(u['epoca'])} | Dice {u['dice']:.4f} | {u['segundos']:.0f}s por época"
        else:
            estado = "preparando"
        print(time.strftime("%H:%M"), f"GPU {gpu}", nb, estado, flush=True)
        if proc.poll() is not None:
            esperar(proc, nb)
            del activos[gpu]
            if cola:
                siguiente = cola.pop(0)
                activos[gpu] = (siguiente, ejecutar(siguiente, gpu))

## 6. Comparación y campeón

In [ ]:
p = ejecutar("05_champion.ipynb")
esperar(p, "05_champion.ipynb")
if Path("artifacts/champion.json").exists():
    print(Path("artifacts/champion.json").read_text())

## 7. Subir todo al repositorio
Sube los notebooks ejecutados, `artifacts/` (pesos, métricas, figuras y `champion.json`) y las arquitecturas.

In [ ]:
%cd /kaggle/working/repo
!git add -A
!git status --short | head -40
!git commit -q -m "Notebooks ejecutados, artefactos y campeón"
!git pull -q --rebase https://{token}@github.com/{REPO}.git main
!git push -q https://{token}@github.com/{REPO}.git HEAD:main
!git log --oneline -3